In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("../data/raw/nifty50_raw.csv", index_col="Date", parse_dates=True)
df.head()

,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,8395.450195,8410.599609,8288.700195,8288.700195,101900
2015-01-05,8378.400391,8445.599609,8363.900391,8407.950195,118200
2015-01-06,8127.350098,8327.849609,8111.350098,8325.299805,172800
2015-01-07,8102.100098,8151.200195,8065.450195,8118.649902,164100
2015-01-08,8234.599609,8243.500000,8167.299805,8191.399902,143800


In [2]:
df["MA5"] = df["Close"].rolling(window=5).mean()
df["MA20"] = df["Close"].rolling(window=20).mean()

In [3]:
df["Daily_Return"] = df["Close"].pct_change()
df["Volatility"] = df["Daily_Return"].rolling(window=10).std()

In [4]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df["RSI14"] = compute_rsi(df["Close"], period=14)

In [5]:
# 1 if next day's close is higher than today's, else 0
df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

In [6]:
df_clean = df.dropna()
print(df_clean.shape)
print(df_clean["Target"].value_counts(normalize=True))  # check class balance vs 55% baseline

out_path = Path("../data/processed/nifty50_features.csv")
df_clean.to_csv(out_path)
print(f"Saved to {out_path.resolve()}")

(2689, 11)
Target
1    0.535515
0    0.464485
Name: proportion, dtype: float64
Saved to C:\Users\Parth Sharma\nifty50-direction-prediction\data\processed\nifty50_features.csv
